In [3]:
import os
import numpy as np
import rasterio
from tqdm import tqdm
import matplotlib.pyplot as plt

In [4]:
# ===============================
# 1. PREPROCESSING FUNCTIONS
# ===============================

def extract_features_from_sar(img):
    vv = img[0].astype(np.float32)
    vh = img[1].astype(np.float32)
    ratio = vv / (vh + 1e-8)
    return np.stack([vv, vh, ratio], axis=0)

def global_normalize(features):
    mean = np.mean(features)
    std = np.std(features)
    return (features - mean) / (std + 1e-8)

In [6]:
# ==============================================================
# 3. CONSOLIDATED BACKGROUND LOOP (LOOKALIKES & NO-OIL)
# ==============================================================

# --- STEP 1: MANUALLY UPDATE THIS PATH FOR EACH RUN ---
# RUN 1 (Lookalikes): Use "Sentinel-1 SAR Oil spill image train . Part II/01_Train_Val_Lookalike_images/Lookalike"
# RUN 2 (No-Oil):    Use "Sentinel-1 SAR Oil spill image train . Part II/01_Train_Val_No_Oil_Images"
MAIN_DIR = os.getcwd()
CURRENT_CATEGORY_PATH = "Sentinel-1 SAR Oil spill image train . Part II/01_Train_Val_No_Oil_Images/No_oil"
TARGET_IMAGES = 50 

# --- STEP 2: SHARED OUTPUT (Class 0) ---
SAVE_IMG  = os.path.join(MAIN_DIR, "background_preprocessed", "images")
SAVE_MASK = os.path.join(MAIN_DIR, "background_preprocessed", "masks")

os.makedirs(SAVE_IMG, exist_ok=True)
os.makedirs(SAVE_MASK, exist_ok=True)

# --- STEP 3: SMART COUNTER (Prevents Overwriting) ---
# This counts existing files and starts indexing from the next available number
if os.path.exists(SAVE_IMG):
    existing_files = [f for f in os.listdir(SAVE_IMG) if f.endswith('.npy')]
    idx = len(existing_files) 
else:
    idx = 0

# Full path to your TIF images
full_input_path = os.path.join(MAIN_DIR, CURRENT_CATEGORY_PATH)

if not os.path.exists(full_input_path):
    print(f"⚠️ ERROR: Path not found: {full_input_path}")
else:
    all_files = sorted([f for f in os.listdir(full_input_path) if f.endswith('.tif')])
    processed_count = 0

    print(f"Processing images from: {os.path.basename(CURRENT_CATEGORY_PATH)}")
    print(f"Starting saving at index: {idx:05d}")

    for fname in tqdm(all_files):
        if processed_count >= TARGET_IMAGES:
            break

        img_path = os.path.join(full_input_path, fname)
        with rasterio.open(img_path) as src:
            img = src.read()

        # Feature Extraction (VV, VH, Ratio)
        features = extract_features_from_sar(img)
        features = global_normalize(features)

        # Patching Logic: 8x8 = 64 patches per 2048x2048 image
        for i in range(0, 2048, 256):
            for j in range(0, 2048, 256):
                p_img = features[:, i:i+256, j:j+256]
                
                # Every patch in these folders is Class 0 (Background)
                # We manually create a zero mask to satisfy training requirements
                p_mask = np.zeros((256, 256), dtype=np.uint8) 
                
                # Save the pair
                np.save(os.path.join(SAVE_IMG, f"{idx:05d}.npy"), p_img)
                np.save(os.path.join(SAVE_MASK, f"{idx:05d}.npy"), p_mask)
                idx += 1

        processed_count += 1

    print(f"✅ Run Finished. Total background patches in folder: {idx}")

Processing images from: No_oil
Starting saving at index: 03200


  7%|█████▉                                                                            | 50/685 [00:22<04:51,  2.18it/s]

✅ Run Finished. Total background patches in folder: 6400
